In [ ]:
# Imports 

import os
from PIL import Image
import cv2
import numpy as np
from gtda.homology import CubicalPersistence
from gtda.diagrams import PersistenceImage
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

In [ ]:
path = 'Example/path'

In [ ]:
def extract_features(image_path):
    img = np.array(Image.open(image_path).convert("RGB"))
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    raw = gray.astype(np.float32) / 255.0
    grad = gradient_magnitude(gray)
    features = []

    for data in [raw, grad]:
        diag = cp.fit_transform([data])
        pim = pi.fit_transform(diag)
        features.append(pim.flatten())

    return np.concatenate(features)


def gradient_magnitude(image):
    gx = cv2.Sobel(image, cv2.CV_32F, 1, 0)
    gy = cv2.Sobel(image, cv2.CV_32F, 0, 1)
    mag = np.sqrt(gx**2 + gy**2)
    return mag / (mag.max() + 1e-8)


def process_dataset_in_batches(data_dir, save_dir, batch_size=50):
    os.makedirs(save_dir, exist_ok=True)

    classes = {"REAL": 0, "FAKE": 1}

    X_batch = []
    y_batch = []
    batch_idx = 0

    for class_name, label in classes.items():
        class_path = os.path.join(data_dir, class_name)

        for img_name in os.listdir(class_path):
            img_path = os.path.join(class_path, img_name)

            try:
                feats = extract_features(img_path)
                X_batch.append(feats)
                y_batch.append(label)
            except Exception as e:
                print(f"Skipping {img_name}: {e}")
                continue

            if len(X_batch) >= batch_size:
                save_path = os.path.join(save_dir, f"batch_{batch_idx}.npz")

                np.savez(
                    save_path,
                    X=np.array(X_batch),
                    y=np.array(y_batch)
                )

                print(f"Saved {save_path}")

                X_batch, y_batch = [], []
                batch_idx += 1

    # Save remaining
    if X_batch:
        save_path = os.path.join(save_dir, f"batch_{batch_idx}.npz")
        np.savez_compressed(save_path, X=np.array(X_batch), y=np.array(y_batch))
        print(f"Saved final {save_path}")


In [ ]:
cp = CubicalPersistence(homology_dimensions=[0, 1])
pi = PersistenceImage()

train_dir = path + "/train"
test_dir = path + "/test"

process_dataset_in_batches(
    train_dir,
    save_dir="train_batches",
    batch_size=500
)


In [ ]:
process_dataset_in_batches(
    test_dir,
    save_dir="test_batches",
    batch_size=500
)

In [ ]:
path2 = 'example/path'

def get_batches(i):
    batches = [f'batch_{i}.npz', f'batch_{i}.npz',f'batch_{i + 100}.npz', f'batch_{i+ 101}.npz'] 
    # Make sure you are getting batches with different labels
    X_list, y_list = [], []
    for file in batches:
        X, y = load_npz(path2 + file)
        #print(X)
        X_list.append(X)
        y_list.append(y)
    return np.vstack(X_list), np.concatenate(y_list)


def load_npz(file):
    data = np.load(file)
    X = data['X'] 
    y = data['y']
    return X.astype(np.float32), y




In [ ]:
# Random Forest


# Obsolite, early try at training random forest, differ to multiple models methods below
rf = RandomForestClassifier(
    n_estimators=0,
    warm_start=True,
    n_jobs=1  # important for memory
)

trees_per_batch = 10


for i in range(0, 50): 
    X_batch, y_batch = get_batches(2*i)
    rf.n_estimators += trees_per_batch
    rf.fit(X_batch, y_batch)
    print(f'Trained batch number {i}')
    del X_batch, y_batch

In [ ]:
path3 = 'example/path'
def get_test_batches():
    X_list, y_list = [], []
    for img_name in os.listdir(path3):
        img_path = os.path.join(path3, img_name)
        X, y = load_npz(img_path)
        X_list.append(X)
        y_list.append(y)
    return np.vstack(X_list), np.concatenate(y_list)

In [ ]:
X_test, y_test = get_test_batches()

y_pred = rf.predict(X_test)

print("\nAccuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))



In [ ]:
#del X_test, y_test

In [ ]:
# Multiiple Model Random Forest
import numpy as np
import random
import gc
from collections import deque
from joblib import Parallel, delayed
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import random


In [ ]:
def sample_params():
    return {
        "max_depth": random.choice([5, 10, 15]),
        "max_features": random.choice(["sqrt", "log2", 0.5, 0.8]),
        "min_samples_split": random.choice([5, 10, 15])
    }

# Stochasically randomize parameters
def mutate_params(params):
    new_params = params.copy()
    
    if random.random() < 0.5:
        new_params["max_depth"] = int(parent["params"]["max_depth"] * random.uniform(0.8, 1.2))
    
    if random.random() < 0.5:
        new_params["max_features"] = random.choice(["sqrt", "log2", 0.5, 0.8])
    
    if random.random() < 0.5:
        new_params["min_samples_split"] = int(parent['params']['min_samples_split'] * random.uniform(.5, 1.5))
    
    return new_params

In [ ]:
def get_batches(i):
    batches = [f'batch_{i}.npz', f'batch_{i}.npz',f'batch_{i + 100}.npz', f'batch_{i+ 101}.npz'] 
    # Make sure you are getting batches with different labels
    X_list, y_list = [], []
    for file in batches:
        X, y = load_npz(path2 + file)
        #print(X)
        X_list.append(X)
        y_list.append(y)
    return np.vstack(X_list), np.concatenate(y_list)


# Function to evaluate models as they are running
def evaluate_model(model):
    y_true, y_pred = [], []
    X_batch, y_batch = get_test_batches_rand()
    #print(X_batch)
    preds = model.predict(X_batch)
    y_true.extend(y_batch)
    y_pred.extend(preds)
    a = accuracy_score(y_true, y_pred)
    return a

def get_test_batches_rand(n = 5):
    X_list, y_list = [], []
    for i in range(n):
        num = random.randint(0, 39)
        img_path = os.path.join(path3, f"batch_{num}.npz")
        X, y = load_npz(img_path)
        X_list.append(X)
        y_list.append(y)
    # print(X_list, y_list)
    return np.vstack(X_list), np.concatenate(y_list)

In [ ]:

EVAL_INTERVAL = 5
trees_per_batch = 4
BUFFER_SIZE = 5
N_MODELS = 6
MIN_TREES_BEFORE_KILL = 20
KILL_THRESHOLD = 0.5  

buffer_X = deque(maxlen=BUFFER_SIZE)
buffer_y = deque(maxlen=BUFFER_SIZE)


models = []
for _ in range(N_MODELS):
    params = sample_params()
    rf = RandomForestClassifier(
        n_estimators=0,
        warm_start=True,
        n_jobs=1,
        **params
    )
    models.append({
        "model": rf,
        "params": params,
        "score": 0,
        "dead": False
    })
print(models)

In [ ]:

def train_model(m, X, y):
    if m["dead"]:
        return m
    
    rf = m["model"]
    rf.n_estimators += trees_per_batch
    rf.fit(X, y)
    
    return m



for i in range(50):
    print(f"\nIteration {i}")
    
    X_batch, y_batch = get_batches(2 * i)
    X_batch = X_batch.astype(np.float32)
    
    buffer_X.append(X_batch)
    buffer_y.append(y_batch)
    

    X_train = np.vstack(buffer_X)
    y_train = np.concatenate(buffer_y)
    
    models = Parallel(n_jobs=-1)(
        delayed(train_model)(m, X_train, y_train)
        for m in models
    )
    

    if i % EVAL_INTERVAL == 0:
        print("Evaluating models...")
        
        for m in models:
            if not m["dead"]:
                m["score"] = evaluate_model(m["model"])
        
        models.sort(key=lambda x: x["score"], reverse=True)
        
        print("Top score:", models[0]["score"])
        
        for m in models:
            if m["model"].n_estimators >= MIN_TREES_BEFORE_KILL:
                if m["score"] < KILL_THRESHOLD:
                    m["dead"] = True
        
        num_replace = len(models) // 2
        
        for j in range(num_replace):
            parent = models[j]
            
            new_params = mutate_params(parent["params"])
            
            new_model = RandomForestClassifier(
                n_estimators=0,
                warm_start=True,
                n_jobs=1,
                **new_params
            )
            
            models[-(j+1)] = {
                "model": new_model,
                "params": new_params,
                "score": 0,
                "dead": False
            }
    
    # ---- cleanup ----
    del X_batch, y_batch, X_train, y_train
    gc.collect()

In [ ]:
import joblib
import os

save_dir = "rf_models"
os.makedirs(save_dir, exist_ok=True)

for i, m in enumerate(models):
    model = m["model"]
    params = m["params"]
    score = m["score"]
    
    filename = f"{save_dir}/model_{i}_score_{score:.4f}.joblib"
    
    joblib.dump({
        "model": model,
        "params": params,
        "score": score
    }, filename)
    
    print(f"Saved: {filename}")

In [ ]:
import joblib
import pandas as pd

path3 = 'Example/path'
def get_test_batches():
    X_list, y_list = [], []
    for img_name in os.listdir(path3):
        img_path = os.path.join(path3, img_name)
        X, y = load_npz(img_path)
        X_list.append(X)
        y_list.append(y)
    return np.vstack(X_list), np.concatenate(y_list)


X_test, y_test = get_test_batches()





models_path = "example/path"


models = []

for m in os.listdir(models_path):
    model = joblib.load("rf_models/" + m)
    models.append(model)

all_models = pd.DataFrame()
for i, m in enumerate(models):
    model = m["model"]
    y_pred = model.predict(X_test)
    all_models[f'model{i}'] = y_pred
    print("\nAccuracy:", accuracy_score(y_test, y_pred))
    print("\nClassification Report:\n")
    print(classification_report(y_test, y_pred))



print("\n\n\n\n\n")
all_models['med'] = np.round(all_models.median(axis = 1))
print("\nAccuracy:", accuracy_score(y_test, all_models['med']))
print("\nClassification Report:\n")

print(classification_report(y_test, all_models['med']))

In [ ]:
del X_test, y_test

In [ ]:
X_list = []
y_list = []
for i in range(20):
    X, y = load_npz(f"train_batches/batch_{i}.npz")
    X_2, y_2 = load_npz(f"train_batches/batch_{i + 100}.npz")
    X_list.append(X)
    y_list.append(y)
    X_list.append(X_2)
    y_list.append(y_2)
X_list = np.vstack(X_list)
y_list = np.concatenate(y_list)


idx = np.random.permutation(len(X_list))
X_list = X_list[idx]
y_list = y_list[idx]


In [ ]:
def train_model(m, X, y):
    if m["dead"]:
        return m
    
    rf = m["model"]
    rf.n_estimators += trees_per_batch
    rf.fit(X, y)
    
    return m


def get_batches(i):
    batches = [f'batch_{i}.npz', f'batch_{i}.npz',f'batch_{i + 100}.npz', f'batch_{i+ 101}.npz'] 
    # Make sure you are getting batches with different labels
    X_list, y_list = [], []
    for file in batches:
        X, y = load_npz(path4 + file)
        #print(X)
        X_list.append(X)
        y_list.append(y)
    return np.vstack(X_list), np.concatenate(y_list)


def sample_params():
    return {
        "max_depth": random.choice([5, 10, 15]),
        "max_features": random.choice(["sqrt", "log2", 0.5, 0.8]),
        "min_samples_split": random.choice([5, 10, 15])
    }

# Stochasically randomize parameters
def mutate_params(params):
    new_params = params.copy()
    
    if random.random() < 0.5:
        new_params["max_depth"] = int(parent["params"]["max_depth"] * random.uniform(0.8, 1.2))
    
    if random.random() < 0.5:
        new_params["max_features"] = random.choice(["sqrt", "log2", 0.5, 0.8])
    
    if random.random() < 0.5:
        new_params["min_samples_split"] = int(parent['params']['min_samples_split'] * random.uniform(.5, 1.5))
    
    return new_params

In [ ]:
EVAL_INTERVAL = 4
trees_per_batch = 10
BUFFER_SIZE = 10
N_MODELS = 8
MIN_TREES_BEFORE_KILL = 20
KILL_THRESHOLD = 0.5  

buffer_X = deque(maxlen=BUFFER_SIZE)
buffer_y = deque(maxlen=BUFFER_SIZE)

models = []
for _ in range(N_MODELS):
    params = sample_params()
    rf = RandomForestClassifier(
        n_estimators=0,
        warm_start=True,
        n_jobs=1,
        **params
    )
    models.append({
        "model": rf,
        "params": params,
        "score": 0,
        "dead": False
    })
print(models)


def evaluate_model(model):
    y_true, y_pred = [], []
    X_batch, y_batch = X_test, y_test
    #print(X_batch)
    preds = model.predict(X_batch)
    y_true.extend(y_batch)
    y_pred.extend(preds)
    a = accuracy_score(y_true, y_pred)
    return a

def get_test_batches_rand(n = 40):
    X_list, y_list = [], []
    for i in range(n):
        num = i
        img_path = os.path.join(path3, f"batch_{num}.npz")
        X, y = load_npz(img_path)
        X_list.append(X)
        y_list.append(y)
    # print(X_list, y_list)
    return np.vstack(X_list), np.concatenate(y_list)


X_test, y_test = get_test_batches_rand()


meta = pd.DataFrame(columns = ['epochs', 'score', 'model_params'])

for i in range(50):
    print(f"\nIteration {i}")
    
    X_batch, y_batch = X_list[i:i+10], y_list[i:i+10]
    X_batch = X_batch.astype(np.float32)
    
    buffer_X.append(X_batch)
    buffer_y.append(y_batch)
    
    X_train = np.vstack(buffer_X)
    y_train = np.concatenate(buffer_y)
    
    models = Parallel(n_jobs=-1)(
        delayed(train_model)(m, X_train, y_train)
        for m in models
    )
    
    if i % EVAL_INTERVAL == 0:
        print("Evaluating models...")
        
        for m in models:
            if not m["dead"]:
                m["score"] = evaluate_model(m["model"])
                meta.loc[len(meta)] = [i, m["score"], m["params"]]
        
        models.sort(key=lambda x: x["score"], reverse=True)
        
        print("Top score:", models[0]["score"])
        
        for m in models:
            if m["model"].n_estimators >= MIN_TREES_BEFORE_KILL:
                if m["score"] < KILL_THRESHOLD:
                    m["dead"] = True
        
        num_replace = len(models) // 2
        
        for j in range(num_replace):
            parent = models[j]
            
            new_params = mutate_params(parent["params"])
            
            new_model = RandomForestClassifier(
                n_estimators=0,
                warm_start=True,
                n_jobs=1,
                **new_params
            )
            
            models[-(j+1)] = {
                "model": new_model,
                "params": new_params,
                "score": 0,
                "dead": False
            }
    
    del X_batch, y_batch, X_train, y_train
    gc.collect()

In [ ]:
meta

In [ ]:
'outputs/hypertuning.png')
plt.show()